In [ ]:
from pathlib import Path
from hydra import initialize_config_dir, compose
import train
import torch

def load_config(directory_or_checkpoint_path: str):
    path = Path(directory_or_checkpoint_path)
    config_dir = path.parent if path.is_file() else path
    
    with initialize_config_dir(config_dir=str(config_dir.absolute()), version_base=None):
        return compose(config_name="config")

def load_model(checkpoint_path: str, device="cuda:0"):
    cfg = load_config(checkpoint_path)
    model = train.create_model(cfg, mask_seed=-1).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device)["model_state_dict"])
    return model

# checkpoint_path = "outputs/2026-03-28/01-17-58__mlp_mnist__k4vckimm/checkpoint_epoch_50_model_1.pt"
# checkpoint_path = "outputs/2026-03-28/01-57-27__mlp_mnist__33yr8r7q/checkpoint_epoch_50_model_1.pt"
checkpoint_path = "outputs/2026-03-28/01-17-58__mlp_mnist__k4vckimm/checkpoint_epoch_50_model_1.pt"
checkpoint_path = "outputs/2026-03-28/04-03-05__mlp_mnist__gkynkvd7/checkpoint_epoch_50_model_1.pt"
import checkpoint_directories
# checkpoint_path = checkpoint_directories.checkpoint_directories_by_architecture["mlp"]["mlp_symmetry1_kappa0"] + "/checkpoint_epoch_100_model_1.pt"
# cfg = load_config(checkpoint_path)
model = load_model(checkpoint_path)
data_info = train.setup_data_loaders(load_config(checkpoint_path))
model

In [ ]:
cfg.model.mask_params

In [ ]:
import src.utils.rebasin.subspace_coherence
from src.utils.record_activations import MODEL_OUTPUT_RECORDING_POINT, RecordInput


activation_recording_points = [("lins.0", RecordInput), ("lins.1", RecordInput), ("lins.2", RecordInput), ("lins.3", RecordInput), MODEL_OUTPUT_RECORDING_POINT]
coherence_results = (
    src.utils.rebasin.subspace_coherence.compute_representation_subspace_coherences(
        activation_recording_points=activation_recording_points,
        model=model,
        data_loader=data_info["train_loader"],
        target_explained_variance_ratio=0.9,
    )
)
coherence_results

In [ ]:
from typing import Dict

from torch.nn import Linear
from plots import plot_grid
from src.models.mlp import SparseLinear
from src.utils.rebasin.subspace_coherence import SubspaceCoherenceResult, compute_gram_matrices, compute_subspace_coherence_and_anisotropy, estimate_subspace_basis_at_explained_variance
from src.utils.record_activations import ActivationRecordingPoint
import src.utils.record_activations
import src.utils.rebasin.subspace_coherence
from src.utils.record_activations import MODEL_OUTPUT_RECORDING_POINT, RecordInput

target_explained_variance_ratio = 0.9
activation_recording_points = [("lins.0", RecordInput), ("lins.1", RecordInput), ("lins.2", RecordInput), ("lins.3", RecordInput)]
[recorded_activations,] = src.utils.record_activations.record_activations(
    activation_recording_points, models=[model], data_loader=data_info["train_loader"]
)

subspace_basis_by_recording_point = {
    recording_point: estimate_subspace_basis_at_explained_variance(torch.cat(activations), target_explained_variance_ratio)[0]
    for recording_point, activations in recorded_activations.items()
}

named_modules = dict(model.named_modules())
gram_matrices_by_recording_point = {
    recording_point: compute_gram_matrices(subspace_basis, layer=named_modules[recording_point[0]])
    for recording_point, subspace_basis in subspace_basis_by_recording_point.items()
}


def __compute_for_recording_point(recording_point, activations, layer_name=None):
    layer = named_modules[recording_point[0]]
    subspace_basis = estimate_subspace_basis_at_explained_variance(torch.cat(activations), target_explained_variance_ratio)[0]
    gram_matrices = compute_gram_matrices(subspace_basis, layer=layer)
    # - induced_preactivation_coeffs: $a$ in the paper
    effective_weight = (
        layer.effective_weight() if isinstance(layer, SparseLinear)
        else layer.weight if isinstance(layer, Linear)
        else None
    )
    induced_preactivation_coeffs = subspace_basis.T @ effective_weight.T
    fixed_weights = (
        (1 - layer.mask.detach()) * layer.mask_constant * layer.normal_mask if isinstance(layer, SparseLinear)
        else torch.zeros_like(effective_weight) if isinstance(layer, Linear)
        else None
    )
    print(subspace_basis.T.shape, fixed_weights.shape)
    projected_centers = subspace_basis.T @ fixed_weights.T # TODO .T?
    print(projected_centers)
    print(induced_preactivation_coeffs.shape, projected_centers.shape)
    diffs = induced_preactivation_coeffs[:, None, :] - projected_centers[:, :, None]
    # diffs[:, i, j] is a_j - v_i. gram_matrices_pinv[i] is S_i^\dagger. 
    gram_matrices_pinv = torch.linalg.pinv(gram_matrices)
    # return diffs, gram_matrices_pinv
    # mahalanobis_distance = torch.einsum("kbB,bkK,KBb->bB", diffs, gram_matrices_pinv, diffs) ** 0.5
    mahalanobis_distance = torch.einsum("kij,ikK,Kij->ij", diffs, gram_matrices_pinv, diffs) ** 0.5
    print(mahalanobis_distance.shape)
    print(mahalanobis_distance)
    import plotly.express as px
    custom_colorscale = [
        [0.0,  "#ff0000"],   # VERY low → bright red
        [0.15, "#ff0000"],   # keep red for tiny range
        [0.3, "#440154"],   # jump to dark purple
        # [0.3,  "#3b528b"],
        # [0.5,  "#21918c"],
        # [0.8,  "#5ec962"],
        [0.6,  "#3b528b"],
        [1.0,  "#1b125b"]
    ]
    fig = px.imshow(mahalanobis_distance.detach().cpu().numpy(), height=1000, color_continuous_scale=custom_colorscale)
    fig.update_layout(xaxis_title="Source neuron", yaxis_title="Target neuron class", title=f"Realization cost of source neuron wrt. target neuron class {layer_name}")
    return fig
    # print(gram_matrices.shape)

plot_grid([
    [__compute_for_recording_point(*(list(recorded_activations.items())[0]), "(Layer 1)"), __compute_for_recording_point(*(list(recorded_activations.items())[1]), "(Layer 2)")],
    [__compute_for_recording_point(*(list(recorded_activations.items())[2]), "(Layer 3)"), __compute_for_recording_point(*(list(recorded_activations.items())[3]), "(Layer 4)")]
],
width=800,
height=800
)


In [ ]:
list(subspace_basis_by_recording_point.values())[0].shape

In [ ]:
import numpy as np
import plotly.express as px
print(np.array([[0,1,2,3], [5,6,7,8]]).shape)
px.imshow(np.array([[0,1,2,3], [5,6,7,8]]))

# Next steps
`measure_realization_costs.py` script

Don't overengineer. Just take all the directories from last night and analyze them.

To analyze:
- subspace dimensions per layer
- subspace coherence per layer (just for lols)
- realization costs mahalanobis distance matrix per layer
- later: realization costs via optimization

Analysis notebook: join the data with the dirs properties.

In [ ]:
diffs.shape, gram_matrices_pinv.shape

In [ ]:
(diffs[:, 0, 1] @ gram_matrices_pinv[0] @ diffs[:, 0, 1])**0.5

In [ ]:
torch.einsum("kij,ikK,Kij->ij", diffs, gram_matrices_pinv, diffs)**0.5

In [ ]:
(torch.einsum("kij,ikK,Kij->ij", diffs, gram_matrices_pinv, diffs)**0.5)[0,1]

In [ ]:
from collections import namedtuple
import os
base_path =  "outputs/2026-03-28"

Key = namedtuple(
    "Key",
    field_names=["dataset_name", "symmetry", "kappa", "d_intrinsic", "hidden_dim"]
)

dirs = {
    Key(
        f"{cfg.dataset.name}{'-' + str(cfg.dataset.parity_subspace_dataset.modulo_base) if cfg.dataset.name == "parity-subspace-dataset" else ''}",
        cfg.model.symmetry,
        cfg.model.mask_params.default.mask_constant if cfg.model.symmetry == 1 else None,
        (cfg.dataset.parity_subspace_dataset if cfg.dataset.name == "parity-subspace-dataset" else cfg.dataset.gaussian_subspace_dataset if cfg.dataset.name == "gaussian-subspace-dataset" else None).d_intrinsic,
        cfg.model.hidden_dim
    ): f"{base_path}/{dir}"
    for dir in [dir for dir in os.listdir(base_path) if "__mlp_mnist__" in dir]
    for cfg in [load_config(f"{base_path}/{dir}")]
    if Path(f"{base_path}/{dir}/checkpoint_epoch_50_model_1.pt").is_file()
}
dirs

In [ ]:
dirs[Key("gaussian-subspace-dataset", symmetry=1, kappa=0., d_intrinsic=128, hidden_dim=512)]
dirs[Key("gaussian-subspace-dataset", symmetry=0, kappa=None, d_intrinsic=128, hidden_dim=512)]

In [ ]:
import pandas as pd
px.parallel_coordinates(pd.DataFrame([
    k._asdict()
    for k in dirs.keys()
]),
color="kappa"
)

In [ ]:
[a + "/checkpoint_epoch_50_model_1.pt" for a in dirs.values()]

In [ ]:
import json
import pandas as pd
with open("outputs/realization-costs-synthetic_rateSweep.json", "r") as f:
    data = json.load(f)
len(data)
df = pd.DataFrame(
    [dict(
        checkpoint_path=result["checkpoint_path"],
        run_directory=result["run_directory"],
        layer=layer_result["layer"],
        subspace_dimension=layer_result["subspace_dimension"],
        full_dimension=layer_result["full_dimension"],
        explained_variance_ratio=layer_result["explained_variance_ratio"],
        subspace_coherence=layer_result["subspace_coherence"],
        i=i, j=j,
        mahalanobis_distance=layer_result["mahalanobis_distances"][i][j]
    )
    for result in data
    for layer_result in result["realization_cost_results"]
    for i in range(len(layer_result["mahalanobis_distances"]))
    for j in range(len(layer_result["mahalanobis_distances"][i]))
]
)
df["is_diag"] = (df["i"] == df["j"])
df_paths = pd.DataFrame([
    {**properties._asdict(), "run_directory": path} 
    for properties, path in dirs.items()
])
df = df.merge(df_paths, on="run_directory")
df["symmetry_kappa"] = ("symmetry" + df["symmetry"].astype(str) + (
    "_kappa=" + df["kappa"].astype(str)).where(df["kappa"].notna(), ""))
df.columns

In [ ]:
df_agg = df.drop(columns=["i", "j", "run_directory"], axis=1
    ).groupby(["dataset_name", "symmetry_kappa",  "symmetry", "kappa", "d_intrinsic", "hidden_dim", "layer", "checkpoint_path", "is_diag"]
    ).agg("mean").reset_index()

In [ ]:
df_agg

In [ ]:
df_agg["dataset_name"].unique()

In [ ]:
fig = px.box(df_agg[df_agg["symmetry_kappa"] == "symmetry1_kappa=1.0"], x="d_intrinsic", y="mahalanobis_distance", color="is_diag", width=600, height=600, log_y=True)
fig.update_xaxes(type="category")
fig

In [ ]:
import natsort
from plots import plot_grid
def distance_box_plot(layer):
    fig = px.box((df_tmp := df[(df["symmetry_kappa"] == "symmetry1_kappa=1.0") & (df["dataset_name"] == "gaussian-subspace-dataset")
                        & (df["hidden_dim"] == 512)
                        & (df["layer"] == layer)
                        #   & (df_agg["is_diag"])
                        ]),
            x="d_intrinsic", y="mahalanobis_distance", color="is_diag", width=600, height=600, log_y=True, log_x=True, title=f"Realization costs ({layer})", category_orders={"is_diag": [False, True]})
    fig.update_xaxes(type="category", categoryarray=natsort.natsorted(df_tmp["d_intrinsic"].unique()))
    return fig
plot_grid([[distance_box_plot("lins.0"), distance_box_plot("lins.1")], [distance_box_plot("lins.2"), distance_box_plot("lins.3")]], width=600, height=600)

In [ ]:
import natsort
from plots import plot_grid
def distance_box_plot(layer):
    fig = px.box((df_tmp := df[(df["symmetry_kappa"] == "symmetry1_kappa=1.0") & (df["dataset_name"] == "gaussian-subspace-dataset")
                        & (df["d_intrinsic"] == 128)
                        & (df["layer"] == layer)
                        #   & (df_agg["is_diag"])
                        ]),
            x="hidden_dim", y="mahalanobis_distance", color="is_diag", width=600, height=600, log_y=True, log_x=True, title=f"Realization costs ({layer})")
    fig.update_xaxes(type="category", categoryarray=natsort.natsorted(df_tmp["hidden_dim"].unique()))
    return fig
plot_grid([[distance_box_plot("lins.0"), distance_box_plot("lins.1")], [distance_box_plot("lins.2"), distance_box_plot("lins.3")]], width=600, height=600)

In [ ]:
import natsort
from plots import plot_grid
def distance_box_plot(layer):
    fig = px.box((df_tmp := df[(df["symmetry_kappa"] == "symmetry1_kappa=0.0") & (df["dataset_name"] == "gaussian-subspace-dataset")
                        & (df["hidden_dim"] == 512)
                        & (df["layer"] == layer)
                        #   & (df_agg["is_diag"])
                        ]),
            x="d_intrinsic", y="mahalanobis_distance", color="is_diag", width=600, height=600, log_y=True, log_x=True, title=f"Realization costs ({layer})", category_orders={"is_diag": [False, True]})
    fig.update_xaxes(type="category", categoryarray=natsort.natsorted(df_tmp["d_intrinsic"].unique()))
    return fig
plot_grid([[distance_box_plot("lins.0"), distance_box_plot("lins.1")], [distance_box_plot("lins.2"), distance_box_plot("lins.3")]], width=600, height=600)

In [ ]:
import natsort
from plots import plot_grid
def distance_box_plot(layer):
    fig = px.box((df_tmp := df[(df["symmetry_kappa"] == "symmetry1_kappa=0.0") & (df["dataset_name"] == "gaussian-subspace-dataset")
                        & (df["d_intrinsic"] == 128)
                        & (df["layer"] == layer)
                        #   & (df_agg["is_diag"])
                        ]),
            x="hidden_dim", y="mahalanobis_distance", color="is_diag", width=600, height=600, log_y=True, log_x=True, title=f"Realization costs ({layer})")
    fig.update_xaxes(type="category", categoryarray=natsort.natsorted(df_tmp["hidden_dim"].unique()))
    return fig
plot_grid([[distance_box_plot("lins.0"), distance_box_plot("lins.1")], [distance_box_plot("lins.2"), distance_box_plot("lins.3")]], width=600, height=600)

In [ ]:
px.scatter(
    (df_tmp := df_agg[(df_agg["symmetry_kappa"] == "symmetry1_kappa=1.0") & (df_agg["dataset_name"] == "gaussian-subspace-dataset")]),
    x="subspace_dimension", y="mahalanobis_distance",
    # symbol=df_tmp["hidden_dim"].astype("category"),
    color="layer",
    symbol="is_diag",
    width=600, height=600, log_y=True
)

In [ ]:
# px.parallel_coordinates(df_agg[(df_agg["symmetry_kappa"] == "symmetry1_kappa=1.0") & (df_agg["dataset_name"] == "gaussian-subspace-dataset")])
px.parallel_coordinates(df_tmp)

In [ ]:
fig = px.line(
    (df_tmp := df_agg[(df_agg["symmetry_kappa"] == "symmetry1_kappa=1.0") & (df_agg["dataset_name"] == "gaussian-subspace-dataset")
                      & (df_agg["d_intrinsic"] == 128)
                      & (df_agg["is_diag"])]),
    x="hidden_dim", y="mahalanobis_distance",
    symbol=df_tmp["d_intrinsic"].astype("category"),
    color="layer",
    width=600, height=600, log_y=True
)
fig.update_xaxes(type="category")


In [ ]:
fig = px.line(
    (df_tmp := df_agg[(df_agg["symmetry_kappa"] == "symmetry1_kappa=1.0") & (df_agg["dataset_name"] == "gaussian-subspace-dataset")
                      & (df_agg["d_intrinsic"] == 128)
                      & (df_agg["is_diag"])]),
    x="hidden_dim", y="predicted_rate",
    symbol=df_tmp["d_intrinsic"].astype("category"),
    color="layer",
    width=600, height=600, log_y=True
)
fig.update_xaxes(type="category")


In [ ]:
df_agg["predicted_rate"] = df_agg["hidden_dim"] ** (-2/df_agg["subspace_dimension"])
df_agg

In [ ]:
df_tmp

In [ ]:
from collections import namedtuple
import os
base_path =  "outputs/2026-03-28"

Key = namedtuple(
    "Key",
    field_names=["dataset_name", "symmetry", "kappa", "d_intrinsic", "hidden_dim"]
)

dirs_rateSweep = {
    Key(
        f"{cfg.dataset.name}{'-' + str(cfg.dataset.parity_subspace_dataset.modulo_base) if cfg.dataset.name == "parity-subspace-dataset" else ''}",
        cfg.model.symmetry,
        cfg.model.mask_params.default.mask_constant if cfg.model.symmetry == 1 else None,
        (cfg.dataset.parity_subspace_dataset if cfg.dataset.name == "parity-subspace-dataset" else cfg.dataset.gaussian_subspace_dataset if cfg.dataset.name == "gaussian-subspace-dataset" else None).d_intrinsic,
        cfg.model.hidden_dim
    ): f"{base_path}/{dir}"
    for dir in [dir for dir in os.listdir(base_path) if Path(f"{base_path}/{dir}/checkpoint_epoch_50_model_1.pt").is_file() and dir > "22-00-00"]
    for cfg in [load_config(f"{base_path}/{dir}")]
}
dirs_rateSweep
sorted([a + "/checkpoint_epoch_50_model_1.pt" for a in dirs_rateSweep.values()])

In [ ]:
len(dirs_rateSweep)